In [1]:
import pandas as pd
import numpy as np
import pyodbc
import warnings
import os

pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 50)

def run_sql(query):
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=conn)
        warnings.filterwarnings("default", category=UserWarning)
    return df

from openpyxl import load_workbook

def overlay_to_excel(df, filepath, sheet_name, start_row=1, start_col=1, header=True):
    """Write df values into an existing sheet without deleting other content/formulas."""
    if not os.path.exists(filepath):
        df.to_excel(filepath, sheet_name=sheet_name, index=False)
        return
    wb = load_workbook(filepath)
    if sheet_name not in wb.sheetnames:
        wb.create_sheet(sheet_name)
    ws = wb[sheet_name]

    r = start_row
    if header:
        for c_idx, col_name in enumerate(df.columns, start=start_col):
            ws.cell(row=r, column=c_idx, value=col_name)
        r += 1

    for _, row_data in df.iterrows():
        for c_idx, val in enumerate(row_data, start=start_col):
            if isinstance(val, (pd.Timestamp, np.datetime64)):
                val = pd.Timestamp(val).to_pydatetime()
            elif isinstance(val, (np.integer,)):
                val = int(val)
            elif isinstance(val, (np.floating,)):
                val = float(val)
            ws.cell(row=r, column=c_idx, value=val)
        r += 1

    wb.save(filepath)
    wb.close()

with pyodbc.connect("DSN=Redshift_prod_new") as conn:
    conn.cursor().execute("SELECT 1").fetchone()
print("ODBC connection OK")

ODBC connection OK


In [2]:
fc_rh_query = """
SELECT pricing_hurdle_name,
       DATE_TRUNC('week', lcdf.application_received_dtm) AS week_start,
       COUNT(DISTINCT CASE WHEN aspect = 'APPLICATION' THEN lcdf.loan_id END) AS apps,
       COUNT(DISTINCT CASE WHEN aspect = 'CONTRACT' THEN lcdf.loan_id END) AS cons,
       AVG(CASE WHEN aspect = 'APPLICATION'
            THEN CASE WHEN COALESCE(bcall_discount_dollars, acall_discount_dollars) <= 2500
                 AND ABS(COALESCE(bcall_amtfin, acall_amtfin) - lcdf.adj_amount_financed_front) <= 250
                 THEN 1.0000 ELSE 0.0000 END END) AS full_call_rate,
       AVG(CASE WHEN aspect = 'CONTRACT' THEN rehashed * 1.0000 END) AS con_rehash_rate
FROM edwnpi.crm_dealer_dim cdd
LEFT JOIN edwnpi.los_deal_current_fact lcdf ON cdd.dealer_number = lcdf.dealer_number
LEFT JOIN sandbox.rehashes rh ON lcdf.loan_id = rh.appid
WHERE cdd.current_version_flag = 1
  AND lcdf.application_received_dtm >= '2023-01-01'
  AND acall_amtfin IS NOT NULL
  AND pricing_hurdle_name IS NOT NULL
  AND pricing_hurdle_name != 'mROA-KMX'
GROUP BY 1, 2
ORDER BY 1, 2
"""

fc_rh_df = run_sql(fc_rh_query)
fc_rh_df['week_start'] = pd.to_datetime(fc_rh_df['week_start'])
fc_rh_df['month'] = fc_rh_df['week_start'].dt.month
fc_rh_df['day'] = fc_rh_df['week_start'].dt.day
fc_rh_df['lob'] = fc_rh_df['pricing_hurdle_name'].str.replace('mROA-', '', regex=False)

print(f"Full call + rehash data: {len(fc_rh_df)} rows")
print(f"Date range: {fc_rh_df['week_start'].min().date()} to {fc_rh_df['week_start'].max().date()}")
print(f"LOBs: {sorted(fc_rh_df['lob'].unique())}")
fc_rh_df.head(5)

Full call + rehash data: 1067 rows
Date range: 2022-12-26 to 2026-06-01
LOBs: ['AN', 'ENT', 'FLD', 'FRN', 'MCY', 'STG']


,pricing_hurdle_name,week_start,apps,cons,full_call_rate,con_rehash_rate,month,day,lob
0,mROA-AN,2022-12-26,15,1,0.2000,0.0000,12,26,AN
1,mROA-AN,2023-01-02,1304,87,0.2937,0.2826,1,2,AN
2,mROA-AN,2023-01-09,1287,95,0.2843,0.3333,1,9,AN
3,mROA-AN,2023-01-16,1270,70,0.2700,0.3561,1,16,AN
4,mROA-AN,2023-01-23,1305,90,0.3019,0.2500,1,23,AN


In [3]:
conv_temp = """
SELECT loan_id,
       MIN(CASE WHEN function_name = 'workflow_pre_bureau_calculate' THEN created_utc_dtm END) AS first_app_processing,
       MIN(CASE WHEN function_name = 'workflow_calculate_contract' THEN created_utc_dtm END) AS first_contract_processing
INTO #time_received
FROM odsnpi.pricing_service_log psl
WHERE function_name IN ('workflow_pre_bureau_calculate', 'workflow_calculate_contract')
GROUP BY 1
"""

conv_query = """
SELECT ldcf.dealer_pricing_hurdle,
       DATE_TRUNC('week', first_app_processing) AS week_start,
       COUNT(DISTINCT tr.loan_id) AS apps,
       COUNT(CASE WHEN first_contract_processing IS NOT NULL THEN tr.loan_id END) * 1.000
           / NULLIF(COUNT(DISTINCT tr.loan_id), 0) AS conversion
FROM #time_received tr
LEFT JOIN edwnpi.los_deal_current_fact ldcf
    ON ldcf.loan_id = tr.loan_id AND ldcf.aspect = 'APPLICATION'
WHERE tr.first_app_processing >= '2025-06-01'
  AND dealer_pricing_hurdle IN ('mROA-AN', 'mROA-ENT', 'mROA-FLD', 'mROA-FRN', 'mROA-MCY', 'mROA-STG')
  AND DATEDIFF('day', first_app_processing, SYSDATE) >= 7
GROUP BY 1, 2
ORDER BY 1, 2
"""

with pyodbc.connect("DSN=Redshift_prod_new") as conn:
    cursor = conn.cursor()
    cursor.execute(conv_temp)
    conn.commit()
    warnings.filterwarnings("ignore", category=UserWarning)
    conv_df = pd.read_sql_query(sql=conv_query, con=conn)
    warnings.filterwarnings("default", category=UserWarning)

conv_df['week_start'] = pd.to_datetime(conv_df['week_start'])
conv_df['month'] = conv_df['week_start'].dt.month
conv_df['day'] = conv_df['week_start'].dt.day
conv_df['lob'] = conv_df['dealer_pricing_hurdle'].str.replace('mROA-', '', regex=False)

print(f"Conversion data: {len(conv_df)} rows")
print(f"Date range: {conv_df['week_start'].min().date()} to {conv_df['week_start'].max().date()}")
print(f"LOBs: {sorted(conv_df['lob'].unique())}")
conv_df.head(5)

Conversion data: 318 rows
Date range: 2025-05-26 to 2026-05-25
LOBs: ['AN', 'ENT', 'FLD', 'FRN', 'MCY', 'STG']


,dealer_pricing_hurdle,week_start,apps,conversion,month,day,lob
0,mROA-AN,2025-05-26,165,0.060606,5,26,AN
1,mROA-AN,2025-06-02,2624,0.074695,6,2,AN
2,mROA-AN,2025-06-09,2539,0.078771,6,9,AN
3,mROA-AN,2025-06-16,2575,0.081942,6,16,AN
4,mROA-AN,2025-06-23,2610,0.070498,6,23,AN


In [4]:
def compute_seasonality(df, rate_col, weight_col, lob_col, metric_name):
    """Compute monthly seasonality factors and a Feb 15 - Mar 15 special window factor.

    Returns a DataFrame with one row per LOB (+ 'ALL') and columns:
        lob, metric, overall_avg, month_1 .. month_12, feb_mar_factor
    """
    def _weighted_avg(slice_df):
        total_w = slice_df[weight_col].sum()
        if total_w == 0:
            return np.nan
        return (slice_df[rate_col] * slice_df[weight_col]).sum() / total_w

    feb_mar_mask = (
        ((df['month'] == 2) & (df['day'] >= 15)) |
        ((df['month'] == 3) & (df['day'] <= 15))
    )

    lobs = sorted(df[lob_col].unique())
    rows = []

    for group_label in lobs + ['ALL']:
        if group_label == 'ALL':
            g = df
        else:
            g = df[df[lob_col] == group_label]

        overall = _weighted_avg(g)
        row = {'lob': group_label, 'metric': metric_name, 'overall_avg': overall}

        for m in range(1, 13):
            m_slice = g[g['month'] == m]
            m_avg = _weighted_avg(m_slice)
            row[f'month_{m}'] = m_avg / overall if overall and not np.isnan(m_avg) else np.nan

        fm_slice = g[g.index.isin(df.loc[feb_mar_mask].index)]
        fm_avg = _weighted_avg(fm_slice)
        row['feb_mar_factor'] = fm_avg / overall if overall and not np.isnan(fm_avg) else np.nan

        rows.append(row)

    return pd.DataFrame(rows)


fc_seasonality = compute_seasonality(fc_rh_df, 'full_call_rate', 'apps', 'lob', 'Full Call Rate')
rh_seasonality = compute_seasonality(fc_rh_df, 'con_rehash_rate', 'cons', 'lob', 'Booked Contract Rehash Rate')
conv_seasonality = compute_seasonality(conv_df, 'conversion', 'apps', 'lob', 'Conversion Rate')

print(f"Full call seasonality: {len(fc_seasonality)} rows (LOBs + ALL)")
print(f"Rehash seasonality:    {len(rh_seasonality)} rows")
print(f"Conversion seasonality: {len(conv_seasonality)} rows")

Full call seasonality: 7 rows (LOBs + ALL)
Rehash seasonality:    7 rows
Conversion seasonality: 7 rows


In [5]:
month_cols = [f'month_{m}' for m in range(1, 13)]
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

def display_seasonality(df, title, data_start):
    print("=" * 120)
    print(f"  {title}")
    print(f"  Data from: {data_start}")
    print(f"  Factor = month weighted avg / overall weighted avg  (1.0 = no seasonal effect)")
    print("=" * 120)

    display_df = df[['lob', 'overall_avg'] + month_cols + ['feb_mar_factor']].copy()
    display_df.columns = ['LOB', 'Overall Avg'] + month_labels + ['Feb15-Mar15']

    formatted = display_df.copy()
    formatted['Overall Avg'] = formatted['Overall Avg'].map(lambda x: f'{x:.4f}' if pd.notna(x) else '-')
    for col in month_labels + ['Feb15-Mar15']:
        formatted[col] = formatted[col].map(
            lambda x: f'*{x:.4f}*' if pd.notna(x) and abs(x - 1.0) > 0.05
            else (f'{x:.4f}' if pd.notna(x) else '-')
        )

    print(formatted.to_string(index=False))
    print()
    print("  * = deviates more than 5% from 1.0")
    print("-" * 120)
    print()


display_seasonality(fc_seasonality, "FULL CALL RATE -- Monthly Seasonality Factors by LOB",
                    fc_rh_df['week_start'].min().date())

display_seasonality(rh_seasonality, "BOOKED CONTRACT REHASH RATE -- Monthly Seasonality Factors by LOB",
                    fc_rh_df['week_start'].min().date())

display_seasonality(conv_seasonality, "CONVERSION RATE -- Monthly Seasonality Factors by LOB",
                    conv_df['week_start'].min().date())

  FULL CALL RATE -- Monthly Seasonality Factors by LOB
  Data from: 2022-12-26
  Factor = month weighted avg / overall weighted avg  (1.0 = no seasonal effect)
LOB Overall Avg      Jan    Feb      Mar      Apr      May      Jun      Jul      Aug      Sep      Oct      Nov      Dec Feb15-Mar15
 AN      0.3442   1.0038 0.9753   1.0416 *1.1250* *1.0891*   1.0068 *0.9284* *0.9263* *0.9003* *0.9051*   0.9503   0.9969      0.9833
ENT      0.3949 *1.0680* 1.0489 *1.0788*   0.9931 *0.9194*   1.0157   0.9626 *0.9468* *0.9118* *0.9448*   1.0085   1.0069    *1.0630*
FLD      0.2849   0.9924 1.0232 *1.0756*   1.0224   1.0065   0.9670 *0.9494*   0.9576 *0.9408* *0.9453*   0.9892   1.0256      1.0378
FRN      0.1666   0.9719 1.0408 *1.1642* *1.0828*   1.0165   0.9685 *0.8508* *0.8646* *0.8698* *0.9331*   0.9971   0.9860    *1.0854*
MCY      0.2401 *0.8996* 0.9503   1.0146   1.0257   1.0140 *1.0700* *1.0550* *1.1842*   1.0227   0.9505 *0.8735* *0.8401*      0.9787
STG      0.2743 *0.9370* 1.0027 *1.1

In [6]:
output_file = 'seasonality_analysis.xlsx'

export_cols = ['lob', 'metric', 'overall_avg'] + month_cols + ['feb_mar_factor']
rename_map = dict(zip(month_cols, month_labels))
rename_map.update({'lob': 'LOB', 'metric': 'Metric', 'overall_avg': 'Overall Avg', 'feb_mar_factor': 'Feb15-Mar15'})

for df, sheet in [(fc_seasonality, 'Full Call Seasonality'),
                  (rh_seasonality, 'Rehash Seasonality'),
                  (conv_seasonality, 'Conversion Seasonality')]:
    export_df = df[export_cols].rename(columns=rename_map).copy()
    overlay_to_excel(export_df, output_file, sheet)
    print(f"Exported '{sheet}' to {output_file}")

print(f"\nAll seasonality factors written to '{output_file}'")

Exported 'Full Call Seasonality' to seasonality_analysis.xlsx
Exported 'Rehash Seasonality' to seasonality_analysis.xlsx
Exported 'Conversion Seasonality' to seasonality_analysis.xlsx

All seasonality factors written to 'seasonality_analysis.xlsx'
